In [4]:
import pandas as pd

file_path = "/home/user-kp/anugreha/human/GRN/BIOGRID-ALL-4.4.242.tab3.txt"
df = pd.read_csv(file_path, sep="\t", low_memory=False)
df_human = df[(df['Organism ID Interactor A'] == 9606) & (df['Organism ID Interactor B'] == 9606)]
df_gene_gene = df_human[df_human['Experimental System Type'] == 'genetic']

df_gene_gene = df_gene_gene[['Official Symbol Interactor A', 'Official Symbol Interactor B']]

output_file = "/home/user-kp/anugreha/human/GRN/BioGRID_human_interactions.txt"
df_gene_gene.to_csv(output_file, sep="\t", index=False)

print(f"Extracted {len(df_gene_gene)} gene-gene interactions for humans")

Extracted 18698 gene-gene interactions for humans


In [5]:
pip install networkx

/bin/bash: /home/user-kp/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


load gene-gene interaction network

In [27]:
import networkx as nx
import pandas as pd

network_file = "/home/user-kp/anugreha/human/GRN/BioGRID_human_interactions.txt"
df = pd.read_csv(network_file,sep='\t',header =0)
G = nx.Graph()
G.add_edges_from(df.values)
genes_present = set(G.nodes)
print(df.head())


   Gene A  Gene B
0     BCR   HOXA9
1     ATM    TP53
2   NCOR1      AR
3  CTNNB1  CREBBP
4   BRCA1   CREB1


In [28]:

print(f"{len(G.nodes)}")

7073


load SL pairs

In [29]:
SL_pairs_file = "/home/user-kp/anugreha/human/cancer_mutations/CML/CML_SL_table_TS_2.csv"
sl_df = pd.read_csv(SL_pairs_file, sep = ',', header=0)
print(sl_df.head())
print(f"loaded {len(sl_df)} SL pairs")

   Gene A  Gene B
0  CFAP74   KCNT1
1  CFAP74     VCP
2  CFAP74    HTR4
3  CFAP74   S1PR3
4  CFAP74  TNRC6A
loaded 4074694 SL pairs


Compute distance for each SL pair

In [30]:
def compute_path(G, gene1, gene2):
    if gene1 not in genes_present or gene2 not in genes_present:
        return -2
    try:
        return nx.shortest_path_length(G,source=gene1,target=gene2)
    except nx.NetworkXNoPath:
        return -1
    
sl_df["Network_Distance"] = sl_df.apply(lambda row: compute_path(G, row["Gene A"], row["Gene B"]),axis=1)
sl_df.to_csv("/home/user-kp/anugreha/human/cancer_mutations/CML/SL_network_distance_TS2.csv",sep=',',index=False)
print("path calculated and saved!")

path calculated and saved!


In [31]:
valid_SL_df = sl_df[sl_df["Network_Distance"]==1]
valid_SL_df = valid_SL_df.sort_values(by="Network_Distance",ascending=True)
valid_SL_df.to_csv("/home/user-kp/anugreha/human/cancer_mutations/CML/SL_network_distance_filtered_TS2.csv",sep=',',index=False)
print("saved")

saved


In [32]:
count = (sl_df["Network_Distance"]==1).sum()
print(count)

684


In [2]:
import pandas as pd
network_df = pd.read_csv("/home/user-kp/anugreha/human/GRN/shortlisted_gene_pairs_filtered.csv")
downregulated_df = pd.read_csv("/home/user-kp/anugreha/human/cancer_mutations/HCC/HCC_upreg.csv")

downregulated_genes = set(downregulated_df["Symbol"])

def get_downregulated_status(row):
    gene_a_down = row["Gene A"] in downregulated_genes
    gene_b_down = row["Gene B"] in downregulated_genes

    if gene_a_down and gene_b_down:
        return "Both"
    elif gene_a_down:
        return "Gene A"
    elif gene_b_down:
        return "Gene B"
    else:
        return "None"

network_df["Upregulated"] = network_df.apply(get_downregulated_status, axis=1)

network_df.to_csv("/home/user-kp/anugreha/human/GRN/diff_gene_exp_HCC.csv", index=False)

